# Create CCS Awards (Canadian Cancer Society / CCS Research Institute)

Creates Canadian Cancer Society (CCS) research grants and awards from the
competition **funding results** CCS publishes at
https://cancer.ca/en/research/for-researchers/funding-results (one page per
competition, plus a "Partner-led Grants and Awards" page). Four layouts are
parsed by the script: HTML recipient rows (2019-2022 competitions), "table"
PDFs (2016-2019), award-recipient PDFs (2022-2026) and the partner-led
key/value tables. **586 grants and awards, start years 2017-2026, 100%
title/PI/institution/amount (CAD)/start and end year, CAD 253.6M** (local run
2026-10-01).

**Why not the full portfolio:** CCS's "searchable grants database" (all CCS
grants since 1999) was hosted on CIHR's CRIS application
(`webapps.cihr-irsc.gc.ca/funding/Search?p_version=CCS`); CIHR retired CRIS and
the link now redirects to a CIHR notices page. Only ~1,400 of its detail pages
survive in the Wayback Machine (one July 2017 crawl, no grant numbers), so
pre-2016 CCS/CCSRI/NCIC grants are not ingested. Not parsed either: Awards for
Excellence (prizes named in prose), the October 2022 and 2025 Breakthrough Team
Grants (prose / charts only), the 2023 Emerging Scholar Research Grants (lay
summaries without amounts) and the March 2018 Innovation to Impact page (a
one-off layout).

**Scope:** all research pathways are kept, including Research Training Awards
(master's/doctoral/postdoc, `funding_type` fellowship) and the BioCanRx /
Carleton Black and Indigenous undergraduate summer research internships
(partner-led, `funding_type` training). Flagged: the partner-led "CCTG - Major
Program 2022" grant (CAD 30M to the Canadian Cancer Trials Group, programme /
infrastructure funding).

**Co-funding:** partner names are kept in `funder_scheme`
("{programme} (partner: X)" / "(co-funded ...)"). Several competitions are joint
CCS/CIHR calls (2019 Survivorship Team Grants, 2020 CCS/CIHR/BC Spark Grants,
2025 CIHR Team Grants - Biology to Cancer Prevention, whose PDF lists only the CCS
share); CIHR reports its own share of those under F4320334506 in the CIHR ingest.

**Prerequisites:**
- Run `scripts/local/ccs_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/ccs/ccs_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** synthetic and stable,
`CCS-{start year}-{programme slug}-{family-given}` (+ a 6-hex title hash when one
PI has two grants in one programme-year). Works cite CCS's 6-digit grant numbers
(`70xxxx`: 1,347 of 1,840 stub rows on F4320334576), but no results page or PDF
prints them, so 0 stubs collapse.

**Dates:** `start_year` = first fiscal year / first year of the stated duration,
`start_date` = 1 January of it (several pages state "start date ... January 1");
`end_year` = last fiscal year end / duration end; `end_date` NULL.

**Funder:** shipped under the tracker's F-id **F4320334576 Canadian Cancer Society
Research Institute** (CA, ror 017343w90, doi 10.13039/501100000015). Twins: F4320320118
"Canadian Cancer Society" (same ROR, 614 stub rows, 404 with CCS numbers) and
F4320319928 "Canadian Cancer Society" (no ROR, 798 / 306); not mapped, flagged for the
coordinator. CCSRI was CCS's research arm; current grants are announced as CCS.

**Priority:** `564` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ccs_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ccs/ccs_projects.parquet`;

In [ ]:
%sql
SELECT layout, COUNT(*) as grants, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) as total_cad
FROM openalex.awards.ccs_raw GROUP BY layout;

In [ ]:
%sql
DESCRIBE openalex.awards.ccs_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ccs_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320334576 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320334576;

## Step 2: Create CCS Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ccs_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320334576  -- Canadian Cancer Society Research Institute
),

g AS (
    SELECT
        *,
        TRY_CAST(TRY_CAST(start_year AS DOUBLE) AS INT) AS sy,
        TRY_CAST(TRY_CAST(end_year AS DOUBLE) AS INT) AS ey,
        NULLIF(TRIM(partner), '') AS partner_clean,
        -- free-text co-funding notes on competition listings ("(supported by ...)", "*co-funded with ...");
        -- on partner-led rows the note column holds the cancer site, so it is not used there
        CASE WHEN layout != 'partner_table'
             THEN NULLIF(TRIM(REGEXP_REPLACE(note, '^[*(\\s]+|[)\\s]+$', '')), '') END AS cofund_note
    FROM openalex.awards.ccs_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'CAD' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE
        WHEN LOWER(g.programme) RLIKE 'internship|summer research' THEN 'training'
        WHEN LOWER(g.programme) RLIKE 'training award' THEN 'fellowship'
        ELSE 'research'
    END as funding_type,
    CASE
        WHEN g.partner_clean IS NOT NULL THEN CONCAT(TRIM(g.programme), ' (partner: ', g.partner_clean, ')')
        WHEN g.cofund_note IS NOT NULL THEN CONCAT(TRIM(g.programme), ' (', g.cofund_note, ')')
        ELSE TRIM(g.programme)
    END as funder_scheme,
    'ccs_funding_results' as provenance,
    CASE WHEN g.sy IS NOT NULL THEN TRY_TO_DATE(CONCAT(CAST(g.sy AS STRING), '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.sy as start_year,
    g.ey as end_year,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ccs_funding_results' AND priority = 564;

-- Insert into openalex_awards_raw with priority.
-- Priority 564: direct-from-funder ingest of CCS's own funding results.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    564 as priority
FROM openalex.awards.ccs_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids, COUNT(DISTINCT funder_award_id) as distinct_award_ids
FROM openalex.awards.ccs_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is the synthetic CCS-{year}-{programme}-{pi}[-{hash}] form.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^CCS-20[0-9]{2}-[a-z0-9-]+$' THEN 1 ELSE 0 END) as well_formed
FROM openalex.awards.ccs_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.ccs_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.ccs_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.ccs_awards
GROUP BY funding_type, funder_scheme ORDER BY cnt DESC LIMIT 40;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.ccs_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.ccs_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/amount/PI/start year, CAD only).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_year) as has_end_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.ccs_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ccs_funding_results'
GROUP BY provenance, priority;